# Discovery source と resolution target の lineage を確認する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/u-kitazawa/rhinestone/blob/develop/showcase/04_discovery_lineage.ipynb)

search.ckan.jp が発見した record を `direct` target で Resource へ解決し、発見側と解決側の metadata / provenance が flat merge されず別々に保持されることを確認します。

## Setup

Colab ではこのセルを一度実行します。追加の GIS Runtime は不要です。

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "git+https://github.com/u-kitazawa/rhinestone.git@697e70d812c06e9a0417d1eb86014c0b18e4a8a5",
        ],
        check=True,
    )

## 横断検索から cross-source result を選ぶ

検索語は環境変数で変更できます。結果順は Provider 内の順序であり、Provider 横断の共通 relevance ranking ではありません。公開サービスの状態や結果は変わり得るため、該当 result がない場合は推測で別 URL を組み立てず停止します。

In [ ]:
import os

from rhinestone import configure, sources

query = os.environ.get("RHINESTONE_LINEAGE_QUERY", "河川")
app = configure(sources=(sources.SEARCH_CKAN_JP,))
results = app.search(text=query, limit=5)
cross_source = [item for item in results if item.discovered_by != item.target.source_id]
if not cross_source:
    raise RuntimeError(
        "No cross-source result is currently available; try another explicit query."
    )

result = cross_source[0]
print("Query:", query)
print("Diagnostics:", results.diagnostics)
print("Title:", result.title)
print("Discovered by:", result.discovered_by)
print("Resolution target:", result.target.source_id)
print("Target settings:", sorted(result.target.settings))

## Resource へ解決して両側の record を比較する

`Resource.metadata` と `Resource.provenance` は解決先の record です。cross-source の発見 record は `Resource.discovery` に分離して保持されます。ここではデータ本体を開きません。

In [ ]:
resource = app.resolve(result)
if resource.discovery is None:
    raise RuntimeError("Expected a separate discovery record.")

discovery = resource.discovery
print("Discovery source:", discovery.source_id)
print("Discovery provider:", discovery.provenance.provider)
print("Discovery dataset:", discovery.provenance.dataset_identifier)
print("Discovery resource:", discovery.provenance.resource_identifier)
print("Discovery raw keys:", sorted(discovery.raw_metadata))
print("---")
print("Resolved URI:", resource.uri)
print("Resolved provider:", resource.provenance.provider)
print("Resolved dataset:", resource.provenance.dataset_identifier)
print("Resolved resource:", resource.provenance.resource_identifier)
print("Resolved raw keys:", sorted(resource.source.raw_metadata))
print(
    "AccessPlan:",
    type(resource.access_plan).__name__,
    f"({resource.access_plan.kind})",
)

## 境界

Rhinestone は発見元の record、解決先の Resource、AccessPlan を保持します。公開データの利用、描画、解析、形式変換は downstream Runtime と利用者の責務です。live Provider の可用性は通常 CI の必須条件にしません。